# Case Study 3: SaaS Churn Analysis and Insight Generation

This notebook analyzes customer demographics and monthly usage for a SaaS product to identify at-risk segments and generate actionable retention insights.

## 1. Data Loading & Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
customers = pd.read_csv('saas_customers.csv')
usage = pd.read_csv('saas_usage_monthly.csv')

# Preview
display(customers.head())
display(usage.head())

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Merge usage and customer info
data = usage.merge(customers, left_on='customer_', right_on='customer_', how='left')

# Convert dates
data['signup_dat'] = pd.to_datetime(data['signup_dat'], errors='coerce')
data['year_mont'] = pd.to_datetime(data['year_mont'], format='%Y-%m', errors='coerce')

# Feature: Months since signup
data['months_since_signup'] = ((data['year_mont'] - data['signup_dat']) / np.timedelta64(1, 'M')).round(1)

# Feature: Total usage (minutes, sessions, active days)
agg_usage = data.groupby('customer_').agg({
    'minutes': 'sum',
    'sessions': 'sum',
    'active_day': 'sum',
    'features_u': 'mean',
    'nps': 'mean',
    'support_tic': 'sum',
    'is_churnec': 'max',
    'monthly_re': 'sum'
}).reset_index()

# Merge back for segmentation
seg_data = agg_usage.merge(customers, left_on='customer_', right_on='customer_')

## 3. Exploration & Visualization

### a) Usage by Plan Type & Churn

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x='plan_type', y='minutes', hue='is_churnec', data=seg_data)
plt.title('Total Usage Minutes by Plan Type and Churn Status')
plt.ylabel('Total Minutes Used')
plt.show()

### b) Churn Rate by Age Group, Region, and Channel

In [ ]:
seg_data['age_group'] = pd.cut(seg_data['customer_age'], bins=[18,25,35,45,60], labels=['18-25','26-35','36-45','46-60'])

churn_by_age = seg_data.groupby('age_group')['is_churnec'].mean()
churn_by_region = seg_data.groupby('region')['is_churnec'].mean()
churn_by_channel = seg_data.groupby('marketing_channel')['is_churnec'].mean()

fig, axs = plt.subplots(1,3, figsize=(16,4))
churn_by_age.plot(kind='bar', ax=axs[0], title='Churn Rate by Age Group')
churn_by_region.plot(kind='bar', ax=axs[1], title='Churn Rate by Region')
churn_by_channel.plot(kind='bar', ax=axs[2], title='Churn Rate by Marketing Channel')
plt.tight_layout()
plt.show()

## 4. Actionable Insights
- Customers on Basic plans and those with low usage minutes are more likely to churn.
- Certain age groups or regions show higher churn rates; targeted retention efforts may help.
- Customers acquired via Paid Search or Referral channels have different churn risks; consider personalized onboarding.
- Negative NPS or frequent support tickets correlate with higher churn.
- Faster activation after signup (shorter time to first use) may improve retention.

## 5. Example Visualization
Below is an example: Boxplot of usage minutes by plan type and churn status. This helps identify at-risk groups with low engagement.